# Corrected External Lab Evaluation: `metabolite_name` -> `refmet_name`

This notebook evaluates the tuned SapBERT model against the professor's target task:

```text
input query: metabolite_name
target:      refmet_name, or any retrieved synonym with the same mapped PubChem CID as refmet_name
```

A prediction is counted as correct when a retrieved reference candidate either:

1. exactly matches the normalized `refmet_name`, or
2. shares at least one CID with the normalized `refmet_name` after mapping `refmet_name` into the full CID/synonym reference.

This is different from the earlier lab evaluation, which used lab-name-to-CID rows as the query labels. The earlier notebook is left unchanged.


## Install Dependencies


In [ ]:
import subprocess
import sys

base_packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'duckdb>=1.0',
    'tqdm>=4.66',
    'PyYAML>=6.0',
    'torch>=2.1',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
    'matplotlib>=3.8',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *base_packages])

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu-cu12'])
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu>=1.8'])

print('Installed notebook dependencies.')


## Mount Drive and Locate Repo


In [ ]:
from pathlib import Path
import gc
import json
import os
import shutil
import sys
import time

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer.')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration


In [ ]:
from chem_synonymizer.embed import load_hf_model, pool_outputs
from chem_synonymizer.index import build_faiss_index_from_memmap, faiss_gpu_available
from chem_synonymizer.utils import save_json

# Inputs.
LAB_CSV_PATH = REPO_DIR / 'data' / 'allmetnames_062626.csv'
REFERENCE_CSV_PATH = REPO_DIR / 'data' / 'CID-Synonym-full.csv'
MODEL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'

METABOLITE_COL = 'metabolite_name'
REFMET_COL = 'refmet_name'
CID_COL = 'cid'
NAME_COL = 'name'

# Outputs.
RUN_NAME = 'sapbert_contrastive_refmet_corrected'
RESULTS_RUN_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_refmet_corrected'
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'

# Optional reuse of already-built full-corpus artifacts from the earlier external lab run.
SHARED_ARTIFACT_DIRS = [
    REPO_DIR / 'results' / 'sapbert_contrastive' / 'external_lab_unique_surface' / 'artifacts',
]

SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').exists() else Path('/content')
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / RUN_NAME
WORK_ARTIFACTS_DIR = WORK_DIR / 'artifacts'
PREPARE_DB_PATH = WORK_ARTIFACTS_DIR / 'unique_surface_refmet_corrected.duckdb'
CORPUS_PARQUET_PATH = WORK_ARTIFACTS_DIR / 'unique_surface_corpus.parquet'
CORPUS_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'corpus_embeddings.npy'
QUERY_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'metabolite_query_embeddings.npy'
FAISS_INDEX_PATH = WORK_ARTIFACTS_DIR / 'faiss_ivfpq.index'
NEIGHBOR_INDICES_PATH = WORK_ARTIFACTS_DIR / 'metabolite_neighbor_indices.npy'
NEIGHBOR_SCORES_PATH = WORK_ARTIFACTS_DIR / 'metabolite_neighbor_scores.npy'

# Model and index settings. These match the high-memory Colab/A100 setup used for the previous run.
SEED = 0
MAX_LENGTH = 96
EMBED_DTYPE = 'float16'
POOLING = 'cls'
CORPUS_ENCODE_BATCH_SIZE = 8192
QUERY_ENCODE_BATCH_SIZE = 8192

INDEX_TYPE = 'ivfpq'
NLIST = 16384
NPROBE = 512
TRAIN_SIZE = 3_000_000
ADD_BATCH_SIZE = 250_000
SEARCH_BATCH_SIZE = 8192
K_RETRIEVE = 256
M_PQ = 96
NBITS = 8
USE_FLOAT16_FAISS = True
USE_GPU_FAISS_IF_FITS = True
GPU_INDEX_MEMORY_LIMIT_GB = 74.0

REBUILD_PREPARED_CORPUS = False
REENCODE_CORPUS = False
REENCODE_QUERIES = False
REBUILD_FAISS_INDEX = False
RERUN_SEARCH = False

# Avoid copying huge embedding/index artifacts back to Drive unless explicitly needed.
BACKUP_LARGE_ARTIFACTS_TO_RESULTS = False

for required in [LAB_CSV_PATH, REFERENCE_CSV_PATH, MODEL_DIR]:
    if not required.exists():
        raise FileNotFoundError(required)
for path in [RESULTS_RUN_DIR, RESULTS_ARTIFACTS_DIR, WORK_ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Lab CSV:', LAB_CSV_PATH)
print('Reference CSV:', REFERENCE_CSV_PATH)
print('Model:', MODEL_DIR)
print('Results:', RESULTS_RUN_DIR)
print('Scratch:', WORK_DIR, shutil.disk_usage(SCRATCH_ROOT))


## Helpers


In [ ]:
def normalize_series(values: pd.Series) -> pd.Series:
    return (
        values.fillna('')
        .astype('string')
        .str.normalize('NFKC')
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )


def pipe_split(value: object) -> set[str]:
    if pd.isna(value):
        return set()
    return {part.strip() for part in str(value).split('|') if part.strip()}


def pipe_count(value: object) -> int:
    return len(pipe_split(value))


def has_cid_overlap(candidate_cids: object, gold_cids: object) -> bool:
    return bool(pipe_split(candidate_cids) & pipe_split(gold_cids))


def q(path: Path) -> str:
    return str(path).replace("'", "''")


def artifact_candidates(work_path: Path) -> list[Path]:
    return [RESULTS_ARTIFACTS_DIR / work_path.name] + [d / work_path.name for d in SHARED_ARTIFACT_DIRS]


def parts_dir_candidates(work_path: Path) -> list[Path]:
    return [p.with_suffix(p.suffix + '.parts') for p in artifact_candidates(work_path)]


def restore_artifact(work_path: Path) -> bool:
    if work_path.exists():
        return True
    for candidate in artifact_candidates(work_path):
        if candidate.exists():
            work_path.parent.mkdir(parents=True, exist_ok=True)
            print('Restoring artifact:', candidate, '->', work_path)
            shutil.copy2(candidate, work_path)
            for suffix in ['.meta.json', '.timing.json']:
                sidecar = candidate.with_suffix(candidate.suffix + suffix)
                if sidecar.exists():
                    shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))
            return True
    for parts_dir in parts_dir_candidates(work_path):
        if parts_dir.exists():
            work_path.parent.mkdir(parents=True, exist_ok=True)
            parts = sorted(parts_dir.glob('part-*'))
            if not parts:
                continue
            print(f'Reconstructing {work_path.name} from {len(parts)} parts in {parts_dir}...')
            with work_path.open('wb') as out:
                for part in tqdm(parts, desc=f'RESTORE {work_path.name}'):
                    with part.open('rb') as src:
                        shutil.copyfileobj(src, out, length=64 * 1024 * 1024)
            for suffix in ['.meta.json', '.timing.json']:
                sidecar = parts_dir / f'{work_path.name}{suffix}'
                if sidecar.exists():
                    shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))
            return True
    return False


def backup_artifact(work_path: Path, large: bool = False) -> None:
    if large and not BACKUP_LARGE_ARTIFACTS_TO_RESULTS:
        return
    if not work_path.exists():
        return
    target = RESULTS_ARTIFACTS_DIR / work_path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size == work_path.stat().st_size:
        return
    print('Backing up artifact:', work_path, '->', target)
    shutil.copy2(work_path, target)
    for suffix in ['.meta.json', '.timing.json']:
        sidecar = work_path.with_suffix(work_path.suffix + suffix)
        if sidecar.exists():
            shutil.copy2(sidecar, target.with_suffix(target.suffix + suffix))


def file_fingerprint(path: Path) -> dict:
    return {
        'path': str(path),
        'exists': path.exists(),
        'size': int(path.stat().st_size) if path.exists() else None,
        'mtime': float(path.stat().st_mtime) if path.exists() else None,
    }


def load_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def save_config() -> None:
    save_json(
        {
            'run_name': RUN_NAME,
            'task': 'metabolite_name -> refmet_name corrected external lab evaluation',
            'lab_csv': file_fingerprint(LAB_CSV_PATH),
            'reference_csv': file_fingerprint(REFERENCE_CSV_PATH),
            'model_dir': str(MODEL_DIR),
            'metabolite_col': METABOLITE_COL,
            'refmet_col': REFMET_COL,
            'correct_if': 'candidate normalized name equals normalized refmet_name OR candidate CID overlaps mapped refmet_name CID set',
            'max_length': MAX_LENGTH,
            'pooling': POOLING,
            'embedding_dtype': EMBED_DTYPE,
            'index_type': INDEX_TYPE,
            'nlist': NLIST,
            'nprobe': NPROBE,
            'm_pq': M_PQ,
            'nbits': NBITS,
            'k_retrieve': K_RETRIEVE,
        },
        RESULTS_RUN_DIR / 'config.json',
    )


save_config()


## Build or Restore the Unique Reference Surface Corpus


In [ ]:
def prepare_unique_surface_corpus() -> dict:
    restore_artifact(CORPUS_PARQUET_PATH)
    if CORPUS_PARQUET_PATH.exists() and not REBUILD_PREPARED_CORPUS:
        pf = pq.ParquetFile(CORPUS_PARQUET_PATH)
        summary = {
            'loaded_from_cache': True,
            'unique_surface_rows': int(pf.metadata.num_rows),
            'corpus_parquet_path': str(CORPUS_PARQUET_PATH),
        }
        save_json(summary, RESULTS_RUN_DIR / 'corpus_summary.json')
        print(json.dumps(summary, indent=2))
        return summary

    for path in [PREPARE_DB_PATH, CORPUS_PARQUET_PATH]:
        if path.exists():
            path.unlink()

    con = duckdb.connect(str(PREPARE_DB_PATH))
    con.execute(f'PRAGMA threads={max(1, os.cpu_count() or 8)}')
    con.execute('PRAGMA preserve_insertion_order=false')
    con.execute('DROP TABLE IF EXISTS reference_pairs')
    con.execute('CREATE TABLE reference_pairs (normalized_name VARCHAR, cid VARCHAR, display_name VARCHAR)')

    total_rows = 0
    inserted_rows = 0
    t0 = time.perf_counter()
    reader = pd.read_csv(
        REFERENCE_CSV_PATH,
        usecols=[CID_COL, NAME_COL],
        dtype='string',
        chunksize=2_000_000,
        on_bad_lines='skip',
    )
    for chunk in tqdm(reader, desc='Streaming reference CSV'):
        total_rows += len(chunk)
        chunk = chunk.dropna(subset=[CID_COL, NAME_COL]).copy()
        chunk['normalized_name'] = normalize_series(chunk[NAME_COL])
        chunk = chunk[chunk['normalized_name'].str.len() > 0]
        chunk = chunk.drop_duplicates(['normalized_name', CID_COL])
        if chunk.empty:
            continue
        insert_df = chunk.rename(columns={CID_COL: 'cid', NAME_COL: 'display_name'})[
            ['normalized_name', 'cid', 'display_name']
        ]
        con.register('insert_df', insert_df)
        con.execute('INSERT INTO reference_pairs SELECT normalized_name, cid, display_name FROM insert_df')
        inserted_rows += len(insert_df)
        con.unregister('insert_df')

    con.execute('DROP TABLE IF EXISTS unique_surface_corpus')
    con.execute('''
        CREATE TABLE unique_surface_corpus AS
        SELECT
            row_number() OVER (ORDER BY normalized_name) - 1 AS surface_id,
            normalized_name,
            any_value(display_name) AS display_name,
            string_agg(DISTINCT cid, '|') AS candidate_cids,
            count(DISTINCT cid) AS n_candidate_cids
        FROM reference_pairs
        GROUP BY normalized_name
    ''')
    con.execute(f"COPY unique_surface_corpus TO '{q(CORPUS_PARQUET_PATH)}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    n_corpus = int(con.execute('SELECT COUNT(*) FROM unique_surface_corpus').fetchone()[0])
    con.close()

    summary = {
        'loaded_from_cache': False,
        'reference_rows_read': int(total_rows),
        'inserted_reference_pairs_after_chunk_dedup': int(inserted_rows),
        'unique_surface_rows': int(n_corpus),
        'elapsed_sec': float(time.perf_counter() - t0),
        'corpus_parquet_path': str(CORPUS_PARQUET_PATH),
    }
    save_json(summary, RESULTS_RUN_DIR / 'corpus_summary.json')
    backup_artifact(CORPUS_PARQUET_PATH, large=False)
    print(json.dumps(summary, indent=2))
    return summary


corpus_summary = prepare_unique_surface_corpus()

def get_corpus_display_col() -> str:
    names = set(pq.ParquetFile(CORPUS_PARQUET_PATH).schema_arrow.names)
    if 'display_name' in names:
        return 'display_name'
    if 'example_name' in names:
        return 'example_name'
    raise ValueError(f'Unique surface corpus needs display_name or example_name. Found columns: {sorted(names)}')


CORPUS_DISPLAY_COL = get_corpus_display_col()
print('Corpus display-name column:', CORPUS_DISPLAY_COL)


## Load Corrected Lab Queries and Map `refmet_name` to Reference CIDs


In [ ]:
def load_corrected_lab_queries() -> tuple[pd.DataFrame, dict]:
    lab = pd.read_csv(LAB_CSV_PATH, dtype='string', engine='python', on_bad_lines='skip')
    missing = [col for col in [METABOLITE_COL, REFMET_COL] if col not in lab.columns]
    if missing:
        raise ValueError(f'Missing required lab columns: {missing}. Available columns: {list(lab.columns)}')

    lab = lab.reset_index(drop=True).copy()
    lab['source_row_id'] = np.arange(len(lab), dtype=np.int64)
    lab['metabolite_name'] = lab[METABOLITE_COL].astype('string').str.strip()
    lab['refmet_name'] = lab[REFMET_COL].astype('string').str.strip()
    lab['metabolite_normalized'] = normalize_series(lab['metabolite_name'])
    lab['refmet_normalized'] = normalize_series(lab['refmet_name'])

    has_metabolite = lab['metabolite_normalized'].str.len() > 0
    has_refmet = lab['refmet_normalized'].str.len() > 0
    candidate = lab[has_metabolite & has_refmet].copy()
    candidate = candidate.drop_duplicates(['metabolite_normalized', 'refmet_normalized']).reset_index(drop=True)
    candidate['query_i'] = np.arange(len(candidate), dtype=np.int64)

    con = duckdb.connect()
    con.register('lab_refmets', candidate[['query_i', 'refmet_normalized']])
    mapped = con.execute(f'''
        SELECT
            l.query_i,
            u.normalized_name AS mapped_refmet_normalized,
            u.{CORPUS_DISPLAY_COL} AS mapped_refmet_display_name,
            u.candidate_cids AS refmet_candidate_cids,
            u.n_candidate_cids AS refmet_n_candidate_cids
        FROM lab_refmets l
        LEFT JOIN read_parquet('{q(CORPUS_PARQUET_PATH)}') u
            ON l.refmet_normalized = u.normalized_name
    ''').df()
    con.close()
    out = candidate.merge(mapped, on='query_i', how='left')
    out['refmet_mapped_to_reference'] = out['refmet_candidate_cids'].fillna('').astype(str).str.len() > 0
    out['metabolite_equals_refmet_normalized'] = out['metabolite_normalized'].eq(out['refmet_normalized'])

    summary = {
        'lab_rows': int(len(lab)),
        'rows_with_metabolite_name': int(has_metabolite.sum()),
        'rows_with_refmet_name': int(has_refmet.sum()),
        'rows_with_both_names': int((has_metabolite & has_refmet).sum()),
        'deduplicated_query_pairs': int(len(candidate)),
        'queries_with_refmet_mapped_to_reference': int(out['refmet_mapped_to_reference'].sum()),
        'queries_without_refmet_mapping': int((~out['refmet_mapped_to_reference']).sum()),
        'queries_where_metabolite_equals_refmet': int(out['metabolite_equals_refmet_normalized'].sum()),
    }
    return out, summary


all_corrected_queries, query_summary = load_corrected_lab_queries()
eval_queries = all_corrected_queries[all_corrected_queries['refmet_mapped_to_reference']].reset_index(drop=True).copy()
eval_queries['eval_query_i'] = np.arange(len(eval_queries), dtype=np.int64)
eval_queries.to_csv(RESULTS_ARTIFACTS_DIR / 'corrected_all_query_pairs_with_refmet_mapping.csv', index=False)
eval_queries[['eval_query_i', 'source_row_id', 'metabolite_name', 'refmet_name', 'metabolite_normalized', 'refmet_normalized', 'refmet_candidate_cids']].to_csv(
    RESULTS_ARTIFACTS_DIR / 'corrected_eval_queries.csv',
    index=False,
)
save_json(query_summary, RESULTS_RUN_DIR / 'query_mapping_summary.json')
print(json.dumps(query_summary, indent=2))
display(eval_queries.head(10))


## Encode Corpus and Corrected Query Embeddings


In [ ]:
import torch


def encode_parquet_texts_to_memmap(
    parquet_path: Path,
    text_col: str,
    out_path: Path,
    tokenizer,
    model,
    model_name: str,
    batch_size: int,
    device: str,
    run_key: str,
) -> np.ndarray:
    pf = pq.ParquetFile(parquet_path)
    n = int(pf.metadata.num_rows)
    d = int(model.config.hidden_size)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path = out_path.with_suffix(out_path.suffix + '.meta.json')
    expected_meta = {
        'n': n,
        'd': d,
        'text_col': text_col,
        'model_name': model_name,
        'pooling': POOLING,
        'max_length': int(MAX_LENGTH),
        'dtype': EMBED_DTYPE,
        'normalize': True,
        'run_key': run_key,
    }
    if out_path.exists() and meta_path.exists() and not REENCODE_CORPUS:
        old_meta = load_json_if_exists(meta_path)
        arr = np.load(out_path, mmap_mode='r')
        if old_meta == expected_meta and arr.shape == (n, d):
            print('Using cached corpus embeddings:', out_path)
            return arr

    if out_path.exists():
        out_path.unlink()
    arr = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.float16 if EMBED_DTYPE == 'float16' else np.float32, shape=(n, d))
    autocast_enabled = device.startswith('cuda') and torch.cuda.is_available()
    offset = 0
    t0 = time.perf_counter()
    with torch.inference_mode():
        for record_batch in tqdm(pf.iter_batches(batch_size=int(batch_size), columns=[text_col]), total=max(1, n // int(batch_size)), desc=f'Encoding {text_col}'):
            values = record_batch.column(0).to_pylist()
            texts = [str(value) for value in values]
            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=int(MAX_LENGTH),
                return_tensors='pt',
            ).to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
                out = model(**enc)
                emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            end = offset + len(texts)
            arr[offset:end] = emb.detach().cpu().numpy().astype(arr.dtype, copy=False)
            offset = end
            del enc, out, emb, texts
            if device.startswith('cuda'):
                torch.cuda.empty_cache()
    arr.flush()
    save_json(expected_meta, meta_path)
    save_json({'elapsed_sec': float(time.perf_counter() - t0)}, out_path.with_suffix(out_path.suffix + '.timing.json'))
    return np.load(out_path, mmap_mode='r')


def encode_query_texts_to_memmap(texts: list[str], out_path: Path, tokenizer, model, model_name: str, batch_size: int, device: str) -> np.ndarray:
    n = len(texts)
    d = int(model.config.hidden_size)
    meta_path = out_path.with_suffix(out_path.suffix + '.meta.json')
    expected_meta = {
        'n': n,
        'd': d,
        'model_name': model_name,
        'pooling': POOLING,
        'max_length': int(MAX_LENGTH),
        'dtype': EMBED_DTYPE,
        'normalize': True,
        'run_key': RUN_NAME,
    }
    if out_path.exists() and meta_path.exists() and not REENCODE_QUERIES:
        old_meta = load_json_if_exists(meta_path)
        arr = np.load(out_path, mmap_mode='r')
        if old_meta == expected_meta and arr.shape == (n, d):
            print('Using cached query embeddings:', out_path)
            return arr
    if out_path.exists():
        out_path.unlink()
    arr = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.float16 if EMBED_DTYPE == 'float16' else np.float32, shape=(n, d))
    autocast_enabled = device.startswith('cuda') and torch.cuda.is_available()
    t0 = time.perf_counter()
    with torch.inference_mode():
        for start in tqdm(range(0, n, int(batch_size)), desc='Encoding metabolite_name queries'):
            end = min(start + int(batch_size), n)
            batch = [str(x) for x in texts[start:end]]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=int(MAX_LENGTH), return_tensors='pt').to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
                out = model(**enc)
                emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            arr[start:end] = emb.detach().cpu().numpy().astype(arr.dtype, copy=False)
            del enc, out, emb, batch
            if device.startswith('cuda'):
                torch.cuda.empty_cache()
    arr.flush()
    save_json(expected_meta, meta_path)
    save_json({'elapsed_sec': float(time.perf_counter() - t0)}, out_path.with_suffix(out_path.suffix + '.timing.json'))
    return np.load(out_path, mmap_mode='r')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Embedding device:', device)
tokenizer, hf_model = load_hf_model(str(MODEL_DIR), device)

restore_artifact(CORPUS_EMBEDDINGS_PATH)
corpus_embeddings = encode_parquet_texts_to_memmap(
    CORPUS_PARQUET_PATH,
    text_col='normalized_name',
    out_path=CORPUS_EMBEDDINGS_PATH,
    tokenizer=tokenizer,
    model=hf_model,
    model_name=str(MODEL_DIR),
    batch_size=CORPUS_ENCODE_BATCH_SIZE,
    device=device,
    run_key='external_lab_unique_surface_sapbert_contrastive',
)

query_embeddings = encode_query_texts_to_memmap(
    eval_queries['metabolite_normalized'].astype(str).tolist(),
    QUERY_EMBEDDINGS_PATH,
    tokenizer,
    hf_model,
    str(MODEL_DIR),
    QUERY_ENCODE_BATCH_SIZE,
    device,
)
print('Corpus embeddings:', corpus_embeddings.shape, corpus_embeddings.dtype)
print('Query embeddings:', query_embeddings.shape, query_embeddings.dtype)

backup_artifact(QUERY_EMBEDDINGS_PATH, large=False)
backup_artifact(CORPUS_EMBEDDINGS_PATH, large=True)


## Build or Load FAISS Index


In [ ]:
import faiss

if device.startswith('cuda'):
    print('Releasing transformer model GPU memory before FAISS...')
    del hf_model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


def expected_faiss_meta(n: int, d: int) -> dict:
    nlist_eff = min(max(int(NLIST), 1), max(int(n), 1), max(int(TRAIN_SIZE), 1)) if INDEX_TYPE != 'flat' else int(NLIST)
    return {
        'index_type': INDEX_TYPE,
        'n': int(n),
        'd': int(d),
        'nlist': int(nlist_eff),
        'train_size': int(TRAIN_SIZE),
        'm_pq': int(M_PQ),
        'nbits': int(NBITS),
        'seed': int(SEED),
        'run_key': 'external_lab_unique_surface_sapbert_contrastive',
    }


def move_faiss_index_to_device(cpu_index, faiss_device: str):
    if faiss_gpu_available(faiss, faiss_device):
        gpu_id = int(faiss_device.split(':', 1)[1]) if ':' in faiss_device else 0
        res = faiss.StandardGpuResources()
        opts = faiss.GpuClonerOptions()
        opts.useFloat16 = bool(USE_FLOAT16_FAISS)
        index = faiss.index_cpu_to_gpu(res, gpu_id, cpu_index, opts)
    else:
        index = cpu_index
    if hasattr(index, 'nprobe'):
        index.nprobe = int(NPROBE)
    return index


def load_cached_faiss_index(expected_meta: dict, faiss_device: str):
    restore_artifact(FAISS_INDEX_PATH)
    old_meta = load_json_if_exists(FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.meta.json'))
    if not FAISS_INDEX_PATH.exists() or old_meta != expected_meta:
        return None
    print('Using cached FAISS index:', FAISS_INDEX_PATH)
    return move_faiss_index_to_device(faiss.read_index(str(FAISS_INDEX_PATH)), faiss_device)


n_corpus, dim = corpus_embeddings.shape
ivfpq_codes_gb = n_corpus * M_PQ / (1024 ** 3)
flat_gpu_gb = n_corpus * dim * (2 if USE_FLOAT16_FAISS else 4) / (1024 ** 3)
estimated_index_gb = ivfpq_codes_gb if INDEX_TYPE == 'ivfpq' else flat_gpu_gb
faiss_device = 'cuda' if USE_GPU_FAISS_IF_FITS and estimated_index_gb <= GPU_INDEX_MEMORY_LIMIT_GB else 'cpu'
print('Estimated FAISS index GB:', round(estimated_index_gb, 2))
print('FAISS device:', faiss_device)

if REBUILD_FAISS_INDEX:
    for stale in [FAISS_INDEX_PATH, FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.meta.json'), FAISS_INDEX_PATH.with_suffix(FAISS_INDEX_PATH.suffix + '.timing.json')]:
        if stale.exists():
            stale.unlink()

faiss_expected_meta = expected_faiss_meta(n_corpus, dim)
faiss_index = None if REBUILD_FAISS_INDEX else load_cached_faiss_index(faiss_expected_meta, faiss_device)
if faiss_index is None:
    faiss_index, _ = build_faiss_index_from_memmap(
        xb_memmap=corpus_embeddings,
        index_path=FAISS_INDEX_PATH,
        index_type=INDEX_TYPE,
        nlist=NLIST,
        nprobe=NPROBE,
        train_size=TRAIN_SIZE,
        add_batch_size=ADD_BATCH_SIZE,
        m_pq=M_PQ,
        nbits=NBITS,
        seed=SEED,
        device=faiss_device,
        use_float16=USE_FLOAT16_FAISS,
        run_key='external_lab_unique_surface_sapbert_contrastive',
    )
    backup_artifact(FAISS_INDEX_PATH, large=True)


## Search `metabolite_name` Queries


In [ ]:
def search_queries(index, query_embeddings: np.ndarray, k_retrieve: int, batch_size: int) -> tuple[np.ndarray, np.ndarray]:
    n = len(query_embeddings)
    idx = np.lib.format.open_memmap(NEIGHBOR_INDICES_PATH, mode='w+', dtype=np.int64, shape=(n, int(k_retrieve)))
    scores = np.lib.format.open_memmap(NEIGHBOR_SCORES_PATH, mode='w+', dtype=np.float32, shape=(n, int(k_retrieve)))
    for start in tqdm(range(0, n, int(batch_size)), desc='FAISS search metabolite_name queries'):
        end = min(start + int(batch_size), n)
        xq = np.ascontiguousarray(np.asarray(query_embeddings[start:end], dtype=np.float32))
        d, i = index.search(xq, int(k_retrieve))
        idx[start:end] = i
        scores[start:end] = d
        del xq, d, i
    idx.flush()
    scores.flush()
    return np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r'), np.load(NEIGHBOR_SCORES_PATH, mmap_mode='r')


expected_search_meta = {
    'n_queries': int(len(eval_queries)),
    'k_retrieve': int(K_RETRIEVE),
    'query_embeddings': file_fingerprint(QUERY_EMBEDDINGS_PATH),
    'faiss_index': file_fingerprint(FAISS_INDEX_PATH),
    'corrected_task': True,
}
search_meta_path = NEIGHBOR_INDICES_PATH.with_suffix(NEIGHBOR_INDICES_PATH.suffix + '.meta.json')
old_search_meta = load_json_if_exists(search_meta_path)
if (
    NEIGHBOR_INDICES_PATH.exists()
    and NEIGHBOR_SCORES_PATH.exists()
    and old_search_meta == expected_search_meta
    and not RERUN_SEARCH
):
    neighbor_indices = np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r')
    neighbor_scores = np.load(NEIGHBOR_SCORES_PATH, mmap_mode='r')
    print('Using cached query neighbors:', neighbor_indices.shape)
else:
    neighbor_indices, neighbor_scores = search_queries(faiss_index, query_embeddings, K_RETRIEVE, SEARCH_BATCH_SIZE)
    save_json(expected_search_meta, search_meta_path)
    backup_artifact(NEIGHBOR_INDICES_PATH, large=False)
    backup_artifact(NEIGHBOR_SCORES_PATH, large=False)
print('Neighbors:', neighbor_indices.shape)


## Corrected Evaluation


In [ ]:
def candidate_rows_for_surface_ids(surface_ids: np.ndarray) -> pd.DataFrame:
    ids = pd.DataFrame({'surface_id': np.unique(surface_ids[surface_ids >= 0]).astype(np.int64)})
    con = duckdb.connect()
    con.register('ids', ids)
    out = con.execute(f'''
        SELECT
            ids.surface_id,
            u.normalized_name AS candidate_normalized_name,
            u.{CORPUS_DISPLAY_COL} AS candidate_name,
            u.candidate_cids,
            u.n_candidate_cids
        FROM ids
        LEFT JOIN read_parquet('{q(CORPUS_PARQUET_PATH)}') u
            ON ids.surface_id = u.surface_id
    ''').df()
    con.close()
    return out


def evaluate_corrected(batch_queries: int = 2000, save_top_k: int = 10) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    ranks = np.full(len(eval_queries), fill_value=np.inf, dtype=np.float64)
    exact_ranks = np.full(len(eval_queries), fill_value=np.inf, dtype=np.float64)
    cid_ranks = np.full(len(eval_queries), fill_value=np.inf, dtype=np.float64)
    prediction_rows = []
    top_rows = []

    query_lookup = eval_queries.set_index('eval_query_i')
    for q_start in tqdm(range(0, len(eval_queries), int(batch_queries)), desc='Evaluating corrected hits'):
        q_end = min(q_start + int(batch_queries), len(eval_queries))
        batch_eval_ids = np.arange(q_start, q_end, dtype=np.int64)
        batch_neighbors = np.asarray(neighbor_indices[q_start:q_end])
        batch_scores = np.asarray(neighbor_scores[q_start:q_end])
        candidates = candidate_rows_for_surface_ids(batch_neighbors.reshape(-1))
        cand_map = candidates.set_index('surface_id').to_dict('index')

        for local_i, eval_i in enumerate(batch_eval_ids):
            qrow = query_lookup.loc[int(eval_i)]
            gold_norm = str(qrow['refmet_normalized'])
            gold_cids = qrow['refmet_candidate_cids']
            best_correct = None
            rank1 = None
            for rank0, surface_id in enumerate(batch_neighbors[local_i]):
                surface_id = int(surface_id)
                if surface_id < 0:
                    continue
                cand = cand_map.get(surface_id, {})
                cand_norm = str(cand.get('candidate_normalized_name', ''))
                cand_cids = cand.get('candidate_cids', '')
                exact = cand_norm == gold_norm
                same_cid = has_cid_overlap(cand_cids, gold_cids)
                correct = bool(exact or same_cid)
                row_payload = {
                    'eval_query_i': int(eval_i),
                    'rank': int(rank0 + 1),
                    'score': float(batch_scores[local_i, rank0]),
                    'surface_id': surface_id,
                    'candidate_name': cand.get('candidate_name', ''),
                    'candidate_normalized_name': cand_norm,
                    'candidate_cids': cand_cids,
                    'correct': correct,
                    'exact_refmet_match': bool(exact),
                    'same_refmet_cid': bool(same_cid),
                }
                if rank0 < int(save_top_k):
                    top_rows.append({
                        **row_payload,
                        'source_row_id': int(qrow['source_row_id']),
                        'metabolite_name': qrow['metabolite_name'],
                        'refmet_name': qrow['refmet_name'],
                        'refmet_candidate_cids': gold_cids,
                    })
                if rank0 == 0:
                    rank1 = row_payload
                if exact and np.isinf(exact_ranks[eval_i]):
                    exact_ranks[eval_i] = rank0 + 1
                if same_cid and np.isinf(cid_ranks[eval_i]):
                    cid_ranks[eval_i] = rank0 + 1
                if correct and best_correct is None:
                    best_correct = row_payload
                    ranks[eval_i] = rank0 + 1
            prediction_rows.append({
                'eval_query_i': int(eval_i),
                'source_row_id': int(qrow['source_row_id']),
                'metabolite_name': qrow['metabolite_name'],
                'refmet_name': qrow['refmet_name'],
                'metabolite_normalized': qrow['metabolite_normalized'],
                'refmet_normalized': qrow['refmet_normalized'],
                'refmet_candidate_cids': gold_cids,
                'top1_name': rank1.get('candidate_name', '') if rank1 else '',
                'top1_candidate_cids': rank1.get('candidate_cids', '') if rank1 else '',
                'top1_correct': bool(rank1.get('correct', False)) if rank1 else False,
                'best_correct_rank': None if best_correct is None else int(best_correct['rank']),
                'best_correct_name': '' if best_correct is None else best_correct.get('candidate_name', ''),
                'best_correct_cids': '' if best_correct is None else best_correct.get('candidate_cids', ''),
                'best_correct_exact_refmet_match': False if best_correct is None else bool(best_correct.get('exact_refmet_match', False)),
                'best_correct_same_refmet_cid': False if best_correct is None else bool(best_correct.get('same_refmet_cid', False)),
            })

    def metrics_from_ranks(rank_arr: np.ndarray, prefix: str = '') -> dict:
        finite = np.isfinite(rank_arr)
        n = len(rank_arr)
        out = {f'{prefix}MRR': float(np.sum(np.where(finite, 1.0 / rank_arr, 0.0)) / n)}
        for k in [1, 5, 10, 100, int(K_RETRIEVE)]:
            out[f'{prefix}HitRate@{int(k)}'] = float(np.mean(rank_arr <= int(k)))
        return out

    metrics = {
        'run_name': RUN_NAME,
        'model_key': 'sapbert_contrastive',
        'task': 'metabolite_name_to_refmet_name_corrected',
        'n_lab_rows': int(query_summary['lab_rows']),
        'n_rows_with_both_names': int(query_summary['rows_with_both_names']),
        'n_deduplicated_query_pairs': int(query_summary['deduplicated_query_pairs']),
        'n_evaluated_queries': int(len(eval_queries)),
        'n_refmet_unmapped_excluded': int(query_summary['queries_without_refmet_mapping']),
        'k_retrieve': int(K_RETRIEVE),
        **metrics_from_ranks(ranks),
        **metrics_from_ranks(exact_ranks, prefix='ExactRefmet_'),
        **metrics_from_ranks(cid_ranks, prefix='SameCid_'),
    }
    return metrics, pd.DataFrame(prediction_rows), pd.DataFrame(top_rows)


corrected_metrics, corrected_predictions, corrected_top_hits = evaluate_corrected()
print(json.dumps(corrected_metrics, indent=2))


## Save Corrected Outputs


In [ ]:
corrected_predictions.to_csv(RESULTS_RUN_DIR / 'corrected_refmet_model_predictions.csv', index=False)
corrected_top_hits.to_csv(RESULTS_RUN_DIR / 'corrected_refmet_top_hits.csv', index=False)
save_json(corrected_metrics, RESULTS_RUN_DIR / 'corrected_refmet_metrics.json')

examples = pd.concat(
    [
        corrected_predictions[corrected_predictions['top1_correct']].head(25).assign(example_type='top1_correct'),
        corrected_predictions[~corrected_predictions['top1_correct'] & corrected_predictions['best_correct_rank'].notna()].head(25).assign(example_type='correct_but_not_top1'),
        corrected_predictions[corrected_predictions['best_correct_rank'].isna()].head(25).assign(example_type='not_found_in_top_k'),
    ],
    ignore_index=True,
)
examples.to_csv(RESULTS_RUN_DIR / 'corrected_refmet_examples.csv', index=False)

artifact_manifest = pd.DataFrame(
    [
        {'field': 'config', 'value': str(RESULTS_RUN_DIR / 'config.json')},
        {'field': 'query_mapping_summary', 'value': str(RESULTS_RUN_DIR / 'query_mapping_summary.json')},
        {'field': 'metrics', 'value': str(RESULTS_RUN_DIR / 'corrected_refmet_metrics.json')},
        {'field': 'predictions', 'value': str(RESULTS_RUN_DIR / 'corrected_refmet_model_predictions.csv')},
        {'field': 'top_hits', 'value': str(RESULTS_RUN_DIR / 'corrected_refmet_top_hits.csv')},
        {'field': 'examples', 'value': str(RESULTS_RUN_DIR / 'corrected_refmet_examples.csv')},
    ]
)
artifact_manifest.to_csv(RESULTS_RUN_DIR / 'corrected_refmet_artifact_manifest.csv', index=False)
display(artifact_manifest)
display(examples.head(20))


## Quick Plots


In [ ]:
import matplotlib.pyplot as plt

labels = ['Hit@1', 'Hit@5', 'Hit@10', 'Hit@100']
values = [
    corrected_metrics['HitRate@1'],
    corrected_metrics['HitRate@5'],
    corrected_metrics['HitRate@10'],
    corrected_metrics['HitRate@100'],
]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, values)
ax.set_ylim(0, 1)
ax.set_ylabel('HitRate')
ax.set_title('Corrected refmet_name target evaluation')
for i, value in enumerate(values):
    ax.text(i, value + 0.02, f'{value:.1%}', ha='center')
fig.tight_layout()
plot_path = RESULTS_RUN_DIR / 'corrected_refmet_hitrate.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved plot:', plot_path)
